# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam271/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [51]:
# ML-04 — Warehouse setup
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get HF token from Colab Secret / environment.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# For ML-04, use the March 2026 partition as the mid-panel month.
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("DuckDB connected.")
print("Verification month: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected.
Verification month: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content page for one client on one report date.
Time window: For this contract, I use March 2026 as the mid-panel verification window. The broader warehouse contains daily content-performance history across the available reporting period.

In [52]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Verify the grain
grain_check = con.execute(f"""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT
        report_date || '|' || client_hash_id || '|' || content_hash_id
    ) AS unique_grain_keys
FROM {MARCH}
""").df()

display(grain_check)

assert grain_check.loc[0, "rows_total"] == grain_check.loc[0, "unique_grain_keys"], \
    "Grain check failed: duplicate report_date × client × content rows found."

print("Grain verified: one row = one report_date × client × content.")

,rows_total,unique_grain_keys
0,9841378,9841378


Grain verified: one row = one report_date × client × content.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



**What one row means:** One row represents one content page for one client on one report date.

**Table(s):** This contract uses the `fact_content_daily_performance` table/partition for daily content-performance observations. The client and content identifiers are retained as context fields.

**Time window:** March 2026 is used as the mid-panel verification window. The broader warehouse contains daily content-performance history across the available reporting period.

**What to predict/rank:** The intended task is to predict or rank a future content-performance outcome/proxy using information available at the decision moment. The future outcome is treated as the label and is not used as an input feature.

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, and `ga4_sessions` are candidate historical performance signals when they are available before the decision moment.

**Context:** `report_date`, `client_hash_id`, `content_hash_id`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available` provide date, identifiers, and data-availability context.

**Deliberate exclusion:** Future/outcome-derived fields and any fields measured after the decision moment are excluded because they would leak information that would not be available when making the prediction.


In [53]:
# Section 2 — Verify the fields used in the contract

selected_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
]

field_check = columns_check[
    columns_check["column_name"].isin(selected_fields)
].copy()

display(field_check)

assert set(selected_fields).issubset(set(columns_check["column_name"])), \
    "One or more selected fields are missing."

assert len(field_check) == len(selected_fields), \
    "Expected all 12 selected fields to be present."

print(f"Verified {len(selected_fields)} contract fields.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
10,gsc_avg_position,DOUBLE,YES,None,None,None


Verified 12 contract fields.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain:** The March data contains 9,841,378 rows and 9,841,378 unique `report_date × client_hash_id × content_hash_id` keys, confirming one row represents one report date × client × content observation.

**Row count and date window:** The March partition contains 9,841,378 rows, covering `2026-03-01` through `2026-03-31`.

**Availability:** Of the 9,841,378 March rows, 3,611,061 have GSC data available and 413,966 have GA4 data available, using `IS TRUE` to explicitly check availability.

These checks verify the grain, time window, row count, and data availability of the March slice using the real warehouse data.


In [54]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Verify grain, counts/date span, and availability

# Query 1 — Grain
grain_check = con.execute(f"""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT
        report_date || '|' || client_hash_id || '|' || content_hash_id
    ) AS unique_grain_keys
FROM {MARCH}
""").df()

display(grain_check)

assert grain_check.loc[0, "rows_total"] == grain_check.loc[0, "unique_grain_keys"], \
    "Grain check failed."

print("1) Grain verified.")


# Query 2 — Row count and date span
count_window = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MARCH}
""").df()

display(count_window)

print("2) Row count and date span verified.")


# Query 3 — Availability using IS TRUE
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {MARCH}
""").df()

display(availability_check)

print("3) Availability verified using IS TRUE.")

,rows_total,unique_grain_keys
0,9841378,9841378


1) Grain verified.


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


2) Row count and date span verified.


,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


3) Availability verified using IS TRUE.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This slice has several limitations: history may be unbalanced across clients and content, some early observations may have GSC data only, and overlapping time windows can make observations dependent on nearby periods. Availability also varies across rows, so missing GSC or GA4 data can limit which signals are usable for some observations. These data limitations mean the results should be treated as **observed and directional**, not causal.


In [55]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Verify data limits

limits_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_unavailable,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable
FROM {MARCH}
""").df()

display(limits_check)

print("Data-limit checks completed for the March slice.")

,total_rows,gsc_available,ga4_available,gsc_unavailable,ga4_unavailable
0,9841378,3611061,413966,6230317,9427412


Data-limit checks completed for the March slice.


### Five features and availability

* `gsc_impressions` — available at the decision moment because it is measured from historical Search Console performance.
* `gsc_clicks` — available at the decision moment because it is measured from historical Search Console performance.
* `gsc_avg_position` — available at the decision moment because it summarizes historical Search Console ranking performance.
* `ga4_pageviews` — available at the decision moment because it is measured from historical Analytics activity.
* `ga4_sessions` — available at the decision moment because it is measured from historical Analytics activity.

These five features are used as observed historical signals; future outcome information is excluded.


In [56]:
# Five-feature frame — March 2026

five_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
]

feature_frame = con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM {MARCH}
LIMIT 10
""").df()

display(feature_frame)

assert all(col in feature_frame.columns for col in five_features), \
    "One or more required features are missing."

print(f"Five-feature frame verified: {len(five_features)} features.")

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


Five-feature frame verified: 5 features.


In [57]:
# Leakage trap — inspect available outcome-related fields

outcome_fields = [
    c for c in columns_check["column_name"].tolist()
    if any(term in c.lower() for term in ["imp", "declin", "trend", "target", "label", "outcome"])
]

print("Outcome-related fields found:")
for c in outcome_fields:
    print("-", c)

Outcome-related fields found:
- gsc_impressions


In [58]:
# Leakage trap — inspect March fields that can define an outcome/proxy

print("Relevant March schema fields:")
for c in columns_check["column_name"].tolist():
    print("-", c)

Relevant March schema fields:
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events
- month


In [59]:
# Section 5 — Deliberate leakage trap
# Create an artificial two-class proxy for the leakage demonstration.
# Then intentionally include the label itself as a feature.

leak_data = con.execute(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    CASE
        WHEN ROW_NUMBER() OVER (ORDER BY report_date, client_hash_id, content_hash_id) % 2 = 0
        THEN 1
        ELSE 0
    END AS proxy_label
FROM {MARCH}
LIMIT 10000
""").df()

# Deliberate leakage: the feature directly contains the label.
leak_data["LEAK_proxy_label"] = leak_data["proxy_label"]

print("Rows used:", len(leak_data))
print("Proxy label distribution:")
display(leak_data["proxy_label"].value_counts().sort_index())

leak_accuracy = (
    leak_data["LEAK_proxy_label"] == leak_data["proxy_label"]
).mean()

print(f"Deliberate leakage accuracy: {leak_accuracy:.3f}")

Rows used: 10000
Proxy label distribution:


,count
proxy_label,
0,5000
1,5000


Deliberate leakage accuracy: 1.000


In [60]:
# Section 5 — Remove leakage and keep the honest feature set

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

# Remove the deliberately leaked label-derived feature.
honest_data = leak_data.drop(columns=["LEAK_proxy_label"]).copy()

print("Leaked feature removed:", "LEAK_proxy_label" not in honest_data.columns)
print("Honest features:", honest_features)

# Simple majority-class baseline.
majority_class = honest_data["proxy_label"].mode()[0]
honest_accuracy = (
    honest_data["proxy_label"] == majority_class
).mean()

print(f"Honest baseline accuracy: {honest_accuracy:.3f}")

Leaked feature removed: True
Honest features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Honest baseline accuracy: 0.500


### Leakage lesson

The deliberate leakage experiment produced an accuracy of **1.000** because the feature `LEAK_proxy_label` directly contained the proxy label. After removing this label-derived feature, the honest majority-class baseline was **0.500** on the balanced proxy.

This demonstrates that a feature derived from the outcome can make model performance appear artificially strong. Such a feature would not be available at the decision moment and must therefore be excluded from the final feature set.

**Limitation:** This leakage experiment uses an artificial proxy label only to demonstrate the leakage mechanism; it is not a production prediction target.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.